In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image
from tqdm.auto import tqdm

from diffusers import DDPMScheduler, UNet2DModel

In [ ]:

DATA_ROOT = Path("./data/CelebFaces/img_align_celeba")
IMAGE_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 0
SEED = 42
MAX_IMAGES = 5000  

 
TRAIN_TIMESTEPS = 1000 # количество шагов диффузии
INFERENCE_STEPS = 100 # количество шагов для генерации
SAMPLE_EVERY_EPOCH = True
SAMPLES_PER_EPOCH = 8
FINAL_NUM_SAMPLES = 16
FINAL_SAMPLE_SEED = 123


MODEL_SIZE = "medium"

MODEL_CONFIGS = {
    "small": {
        "block_out_channels": (32, 64, 64),
        "down_block_types": ("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        "up_block_types": ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    },
    "medium": {
        "block_out_channels": (64, 128, 128),
        "down_block_types": ("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        "up_block_types": ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    },
    "large": {
        "block_out_channels": (64, 128, 256),
        "down_block_types": ("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        "up_block_types": ("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    },
}

if MODEL_SIZE not in MODEL_CONFIGS:
    raise ValueError(f"Unknown MODEL_SIZE: {MODEL_SIZE}")

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset path:", DATA_ROOT)
print("Model size:", MODEL_SIZE)
print("Inference steps:", INFERENCE_STEPS)

In [ ]:
class CelebFacesDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None, max_images=None, seed=42):
        self.root_dir = Path(root_dir)
        self.transform = transform

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset path not found: {self.root_dir}")

        self.image_paths = sorted(self.root_dir.glob("*.jpg"))
        if len(self.image_paths) == 0:
            self.image_paths = sorted(self.root_dir.rglob("*.jpg"))

        if max_images is not None and max_images < len(self.image_paths):
            rng = random.Random(seed)
            self.image_paths = rng.sample(self.image_paths, k=max_images)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image


def denormalize(x):
    return (x.clamp(-1, 1) + 1) / 2


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = CelebFacesDataset(
    DATA_ROOT,
    transform=transform,
    max_images=MAX_IMAGES,
    seed=SEED,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Number of images used: {len(dataset)}")

In [ ]:
batch = next(iter(dataloader))
grid = make_grid(denormalize(batch[:16]), nrow=4)

plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Примеры изображений из CelebFaces")
plt.show()

## Простая diffusion-модель

Размер модели можно менять через параметр `MODEL_SIZE`:
- `small` — быстрее и легче;
- `medium` — хороший базовый вариант;
- `large` — тяжелее, но обычно даёт более качественные результаты.



In [ ]:
model_cfg = MODEL_CONFIGS[MODEL_SIZE]

model = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=model_cfg["block_out_channels"],
    down_block_types=model_cfg["down_block_types"],
    up_block_types=model_cfg["up_block_types"],
).to(device)

noise_scheduler = DDPMScheduler(num_train_timesteps=TRAIN_TIMESTEPS)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
@torch.no_grad()
def sample_images(model, scheduler, n=16, num_inference_steps=INFERENCE_STEPS, seed=42):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, generator=generator, device=device)

    scheduler.set_timesteps(num_inference_steps)
    for t in tqdm(scheduler.timesteps, desc=f"Sampling ({num_inference_steps} steps)"):
        noise_pred = model(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample

    return denormalize(x).cpu()


def show_tensor_images(images, title=None, nrow=4, figsize=(6, 6)):
    grid = make_grid(images, nrow=nrow)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()

## Обучение

На каждом шаге:
- берём реальные изображения;
- добавляем к ним случайный шум;
- выбираем случайный момент времени `t`;
- учим модель предсказывать добавленный шум через `MSE`.

Это самая важная идея обучения DDPM.

In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
    for clean_images in progress_bar:
        clean_images = clean_images.to(device)
        noise = torch.randn_like(clean_images)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean_images.size(0),),
            device=device,
        ).long()

        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)
        noise_pred = model(noisy_images, timesteps).sample
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / len(dataloader)
    history.append(avg_loss)
    print(f"Epoch {epoch}: loss = {avg_loss:.4f}")

    if SAMPLE_EVERY_EPOCH:
        samples = sample_images(
            model,
            noise_scheduler,
            n=SAMPLES_PER_EPOCH,
            num_inference_steps=INFERENCE_STEPS,
            seed=SEED + epoch,
        )
        show_tensor_images(samples, title=f"Сэмплы после эпохи {epoch}", nrow=4, figsize=(8, 4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Кривая обучения")
plt.grid(True)
plt.show()

In [ ]:
final_samples = sample_images(
    model,
    noise_scheduler,
    n=FINAL_NUM_SAMPLES,
    num_inference_steps=INFERENCE_STEPS,
    seed=FINAL_SAMPLE_SEED,
)
show_tensor_images(
    final_samples,
    title="Новые лица, сгенерированные по CelebFaces",
    nrow=4,
    figsize=(8, 8),
)

простой U-Net для diffusion


In [ ]:
# СВОЯ АРХИТЕКТУРА


CUSTOM_MODEL_SIZE = MODEL_SIZE
CUSTOM_EPOCHS = EPOCHS
CUSTOM_INFERENCE_STEPS = INFERENCE_STEPS
CUSTOM_SAMPLES_PER_EPOCH = SAMPLES_PER_EPOCH
CUSTOM_FINAL_NUM_SAMPLES = FINAL_NUM_SAMPLES
CUSTOM_FINAL_SEED = FINAL_SAMPLE_SEED


- `SinusoidalTimeEmbedding` — переводит номер шага `t` в вектор признаков;
- `ConvBlock` — обычный сверточный блок;
- `DownBlock` — уменьшает размер карты признаков;
- `UpBlock` — увеличивает размер и использует skip-connections;
- `SimpleUNet` — объединяет всё в один U-Net.



In [ ]:
import torch.nn as nn

In [ ]:
def get_custom_channels(size: str):
    if size == "small":
        return 32, 64, 128
    if size == "medium":
        return 64, 128, 256
    if size == "large":
        return 64, 128, 256
    raise ValueError(f"Unknown CUSTOM_MODEL_SIZE: {size}")


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        factor = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -factor)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(out_channels)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU()
        self.time_proj = nn.Linear(time_dim, out_channels)

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.act(h)

        time_term = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_term

        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)
        return h


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.block = ConvBlock(in_channels, out_channels, time_dim)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x, t_emb):
        h = self.block(x, t_emb)
        return h, self.pool(h)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.block = ConvBlock(out_channels + skip_channels, out_channels, time_dim)

    def forward(self, x, skip, t_emb):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        x = torch.cat([x, skip], dim=1)
        return self.block(x, t_emb)


class SimpleUNet(nn.Module):
    def __init__(self, image_channels=3, base_channels=64, mid_channels=128, bottleneck_channels=256, time_dim=128):
        super().__init__()
        self.time_embedding = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.down1 = DownBlock(image_channels, base_channels, time_dim)
        self.down2 = DownBlock(base_channels, mid_channels, time_dim)
        self.bottleneck = ConvBlock(mid_channels, bottleneck_channels, time_dim)
        self.up1 = UpBlock(bottleneck_channels, mid_channels, mid_channels, time_dim)
        self.up2 = UpBlock(mid_channels, base_channels, base_channels, time_dim)
        self.out = nn.Conv2d(base_channels, image_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        t_emb = self.time_mlp(t_emb)

        skip1, x = self.down1(x, t_emb)
        skip2, x = self.down2(x, t_emb)
        x = self.bottleneck(x, t_emb)
        x = self.up1(x, skip2, t_emb)
        x = self.up2(x, skip1, t_emb)
        return self.out(x)

In [ ]:
custom_base, custom_mid, custom_bottleneck = get_custom_channels(CUSTOM_MODEL_SIZE)

custom_model = SimpleUNet(
    image_channels=3,
    base_channels=custom_base,
    mid_channels=custom_mid,
    bottleneck_channels=custom_bottleneck,
    time_dim=128,
).to(device)

custom_scheduler = DDPMScheduler(num_train_timesteps=TRAIN_TIMESTEPS)
custom_optimizer = torch.optim.AdamW(custom_model.parameters(), lr=LEARNING_RATE)

print("Custom model size:", CUSTOM_MODEL_SIZE)
print(f"Custom model parameters: {sum(p.numel() for p in custom_model.parameters()):,}")

In [ ]:
@torch.no_grad()
def sample_images_custom(model, scheduler, n=16, num_inference_steps=100, seed=42):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, generator=generator, device=device)

    scheduler.set_timesteps(num_inference_steps)
    for t in tqdm(scheduler.timesteps, desc=f"Custom sampling ({num_inference_steps} steps)"):
        timestep_batch = torch.full((n,), int(t), device=device, dtype=torch.long)
        noise_pred = model(x, timestep_batch)
        x = scheduler.step(noise_pred, t, x).prev_sample

    return denormalize(x).cpu()

In [ ]:
custom_history = []

for epoch in range(1, CUSTOM_EPOCHS + 1):
    custom_model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Custom epoch {epoch}/{CUSTOM_EPOCHS}")
    for clean_images in progress_bar:
        clean_images = clean_images.to(device)
        noise = torch.randn_like(clean_images)

        timesteps = torch.randint(
            0,
            custom_scheduler.config.num_train_timesteps,
            (clean_images.size(0),),
            device=device,
        ).long()

        noisy_images = custom_scheduler.add_noise(clean_images, noise, timesteps)
        noise_pred = custom_model(noisy_images, timesteps)
        loss = F.mse_loss(noise_pred, noise)

        custom_optimizer.zero_grad()
        loss.backward()
        custom_optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / len(dataloader)
    custom_history.append(avg_loss)
    print(f"Custom epoch {epoch}: loss = {avg_loss:.4f}")

    if SAMPLE_EVERY_EPOCH:
        custom_samples = sample_images_custom(
            custom_model,
            custom_scheduler,
            n=CUSTOM_SAMPLES_PER_EPOCH,
            num_inference_steps=CUSTOM_INFERENCE_STEPS,
            seed=SEED + epoch,
        )
        show_tensor_images(
            custom_samples,
            title=f" U-Net после эпохи {epoch}",
            nrow=4,
            figsize=(8, 4),
        )

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(custom_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

In [ ]:
custom_final_samples = sample_images_custom(
    custom_model,
    custom_scheduler,
    n=CUSTOM_FINAL_NUM_SAMPLES,
    num_inference_steps=CUSTOM_INFERENCE_STEPS,
    seed=CUSTOM_FINAL_SEED,
)
show_tensor_images(
    custom_final_samples,
    title="Новые лица от самописного U-Net",
    nrow=4,
    figsize=(8, 8),
)

In [ ]:
ADV_CHANNELS = {
    "small": (32, 64, 128, 256),
    "medium": (64, 128, 256, 512),
    "large": (64, 128, 256, 512),
}[CUSTOM_MODEL_SIZE]
ADV_EPOCHS = CUSTOM_EPOCHS
ADV_INFERENCE_STEPS = CUSTOM_INFERENCE_STEPS
ADV_SAMPLES_PER_EPOCH = CUSTOM_SAMPLES_PER_EPOCH
ADV_FINAL_NUM_SAMPLES = CUSTOM_FINAL_NUM_SAMPLES
ADV_FINAL_SEED = CUSTOM_FINAL_SEED
ADV_TIME_DIM = 256
ADV_DROPOUT = 0.1


In [ ]:
class BetterConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, dropout=0.0):
        super().__init__()
        groups = min(8, out_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout2d(dropout)
        self.time_proj = nn.Linear(time_dim, out_channels)

    def forward(self, x, t_emb):
        x = self.act(self.norm1(self.conv1(x)))
        x = x + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        x = self.dropout(x)
        x = self.act(self.norm2(self.conv2(x)))
        return x


class BetterDown(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, dropout=0.0):
        super().__init__()
        self.block1 = BetterConvBlock(in_channels, out_channels, time_dim, dropout)
        self.block2 = BetterConvBlock(out_channels, out_channels, time_dim, dropout)
        self.down = nn.Conv2d(out_channels, out_channels, 4, stride=2, padding=1)

    def forward(self, x, t_emb):
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x, self.down(x)


class BetterUp(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, time_dim, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.block1 = BetterConvBlock(out_channels + skip_channels, out_channels, time_dim, dropout)
        self.block2 = BetterConvBlock(out_channels, out_channels, time_dim, dropout)

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x


class BetterUNet(nn.Module):
    def __init__(self, image_channels=3, channels=(64, 128, 256, 512), time_dim=256, dropout=0.0):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.time_embedding = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.in_block = BetterConvBlock(image_channels, c1, time_dim, dropout)
        self.down1 = BetterDown(c1, c2, time_dim, dropout)
        self.down2 = BetterDown(c2, c3, time_dim, dropout)
        self.down3 = BetterDown(c3, c4, time_dim, dropout)
        self.mid1 = BetterConvBlock(c4, c4, time_dim, dropout)
        self.mid2 = BetterConvBlock(c4, c4, time_dim, dropout)
        self.up1 = BetterUp(c4, c4, c3, time_dim, dropout)
        self.up2 = BetterUp(c3, c3, c2, time_dim, dropout)
        self.up3 = BetterUp(c2, c2, c1, time_dim, dropout)
        self.out = nn.Conv2d(c1, image_channels, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(self.time_embedding(t))
        x0 = self.in_block(x, t_emb)
        s1, x = self.down1(x0, t_emb)
        s2, x = self.down2(x, t_emb)
        s3, x = self.down3(x, t_emb)
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)
        x = self.up1(x, s3, t_emb)
        x = self.up2(x, s2, t_emb)
        x = self.up3(x, s1, t_emb)
        return self.out(x)


In [ ]:
better_model = BetterUNet(
    image_channels=3,
    channels=ADV_CHANNELS,
    time_dim=ADV_TIME_DIM,
    dropout=ADV_DROPOUT,
).to(device)

better_scheduler = DDPMScheduler(num_train_timesteps=TRAIN_TIMESTEPS)
better_optimizer = torch.optim.AdamW(better_model.parameters(), lr=LEARNING_RATE)

print(f"model parameters: {sum(p.numel() for p in better_model.parameters()):,}")


In [ ]:
@torch.no_grad()
def sample_images_better(model, scheduler, n=16, num_inference_steps=100, seed=42):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, generator=generator
    , device=device)

    scheduler.set_timesteps(num_inference_steps)
    for t in tqdm(scheduler.timesteps, desc=f"Better sampling ({num_inference_steps} steps)"):
        timestep_batch = torch.full((n,), int(t), device=device, dtype=torch.long)
        noise_pred = model(x, timestep_batch)
        x = scheduler.step(noise_pred, t, x).prev_sample

    return denormalize(x).cpu()


In [ ]:
better_history = []

for epoch in range(1, ADV_EPOCHS + 1):
    better_model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Better epoch {epoch}/{ADV_EPOCHS}")
    for clean_images in progress_bar:
        clean_images = clean_images.to(device)
        noise = torch.randn_like(clean_images)

        timesteps = torch.randint(
            0,
            better_scheduler.config.num_train_timesteps,
            (clean_images.size(0),),
            device=device,
        ).long()

        noisy_images = better_scheduler.add_noise(clean_images, noise, timesteps)
        noise_pred = better_model(noisy_images, timesteps)
        loss = F.mse_loss(noise_pred, noise)

        better_optimizer.zero_grad()
        loss.backward()
        better_optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / len(dataloader)
    better_history.append(avg_loss)
    print(f"Better epoch {epoch}: loss = {avg_loss:.4f}")

    if SAMPLE_EVERY_EPOCH:
        better_samples = sample_images_better(
            better_model,
            better_scheduler,
            n=ADV_SAMPLES_PER_EPOCH,
            num_inference_steps=ADV_INFERENCE_STEPS,
            seed=SEED + epoch,
        )
        show_tensor_images(better_samples, title=f"Better U-Net after epoch {epoch}", nrow=4, figsize=(8, 4))


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(better_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("U-Net training curve")
plt.grid(True)
plt.show()


In [ ]:
better_final_samples = sample_images_better(
    better_model,
    better_scheduler,
    n=ADV_FINAL_NUM_SAMPLES,
    num_inference_steps=ADV_INFERENCE_STEPS,
    seed=ADV_FINAL_SEED,
)
show_tensor_images(
    better_final_samples,
    title=" U-Net samples",
    nrow=4,
    figsize=(8, 8),
)
